In [38]:
import numpy as np
import pandas as pd
from pandas.io.stata import StataReader
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

Please read the following report:  
Machine Learning for Poverty Prediction: A Comparative Assessment of Classification Algorithms  
https://github.com/worldbank/ML-classification-algorithms-poverty/blob/master/report/ML_Classification_Poverty_Comparative_Assessment_v01.pdf  
See https://github.com/worldbank/ML-classification-algorithms-poverty for more details.  

Use the following classification approaches for predicting poverty:  
Logistic regression, linear discriminant analysis, k-nearest neighbors, decision trees, random forests, naïve Bayes, and support vector machine.  

1. Replicate all the results in the report using six metrics: accuracy, recall, precision, F1, cross-entropy, ROC AUC, and Cohen-Kappa. 

In [81]:
data = {}
dir = Path("data/")
for file in dir.iterdir():
    data[file.name[9:-4]] = pd.read_stata(file, convert_categoricals=False)
print(data)

{'household':                 hid      wta_hh      wta_pop  der_hhsize  poor  hld_dwelloccu  \
0      101010160009  126.560097   759.360596           6     1              1   
1      101010160033  126.560097   506.240387           4     1              1   
2      101010160060  126.560097  1265.600952          10     1              1   
3      101010160068  126.560097  1139.040894           9     1              1   
4      101010160069  126.560097   126.560097           1     0              1   
...             ...         ...          ...         ...   ...            ...   
12266  315556140342  456.450592  1369.351807           3     0              6   
12267  315556140365  456.450592   912.901184           2     0              5   
12268  315556140388  456.450592   912.901184           2     0              6   
12269  315556140411  456.450592  2282.252930           5     1              6   
12270  315556140434  456.450592  3195.154053           7     0              1   

       hld_dw

In [ ]:
for k in data:
    print(f"{k}: {data[k].shape}")
    nan_counts = (data[k].map(pd.isnull)
                          .sum(axis=0)
                          .sort_values(ascending=False))
    nan_cols = nan_counts[(nan_counts > 0) & (nan_counts < 10)].index.values
    data[k] = data[k].dropna(subset=nan_cols)
    print(f"{k}: {data[k].shape}")
    print()

print(data["household"])

household: (12271, 347)


/tmp/ipykernel_119970/548201411.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nan_counts = (data[k].applymap(pd.isnull)


household: (12246, 347)

community: (768, 16)
community: (767, 16)

individual: (56218, 44)


/tmp/ipykernel_119970/548201411.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nan_counts = (data[k].applymap(pd.isnull)
/tmp/ipykernel_119970/548201411.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nan_counts = (data[k].applymap(pd.isnull)


individual: (56218, 44)

expenditure: (12271, 18)
expenditure: (12269, 18)

                hid      wta_hh      wta_pop  der_hhsize  poor  hld_dwelloccu  \
0      101010160009  126.560097   759.360596           6     1              1   
1      101010160033  126.560097   506.240387           4     1              1   
2      101010160060  126.560097  1265.600952          10     1              1   
3      101010160068  126.560097  1139.040894           9     1              1   
4      101010160069  126.560097   126.560097           1     0              1   
...             ...         ...          ...         ...   ...            ...   
12266  315556140342  456.450592  1369.351807           3     0              6   
12267  315556140365  456.450592   912.901184           2     0              5   
12268  315556140388  456.450592   912.901184           2     0              6   
12269  315556140411  456.450592  2282.252930           5     1              6   
12270  315556140434  456.450592  

/tmp/ipykernel_119970/548201411.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nan_counts = (data[k].applymap(pd.isnull)


In [68]:
print(data["individual"]["poor"].sum()/data["individual"].shape[0])
# print(data["individual"])
print(data["household"]["poor"].sum()/data["household"].shape[0])
# print(data["household"])

0.5243694190472802
0.451307961861299


In [71]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(data["household"].drop(columns=["poor"]), data["household"]["poor"], test_size=0.2, random_state=42)



2. Considering the assumptions behind these approaches, explain why specific approaches do better than others.

3. How related are different evaluation metrics? Considering the nature and domain of the data, what phenomenon are these metrics measuring? How can you explain these results to a person who is not an expert in ML?